In [ ]:
# ==========================================
# Convert Part B .mat → .h5 Density Maps
# ==========================================

import os
import numpy as np
import scipy.io
import h5py
from scipy.ndimage import gaussian_filter
from PIL import Image

train_img_dir = "/content/drive/MyDrive/CrowdCounting/dataset/ShanghaiTech/part_B/train_data/images"
train_gt_dir  = "/content/drive/MyDrive/CrowdCounting/dataset/ShanghaiTech/part_B/train_data/ground-truth"

print("Generating density maps for Part B train set...")

for img_name in sorted(os.listdir(train_img_dir)):

    img_path = os.path.join(train_img_dir, img_name)
    img = Image.open(img_path)
    width, height = img.size

    mat_name = "GT_" + img_name.replace(".jpg", ".mat")
    mat_path = os.path.join(train_gt_dir, mat_name)

    mat = scipy.io.loadmat(mat_path)
    points = mat["image_info"][0][0][0][0][0]

    density = np.zeros((height, width), dtype=np.float32)

    for point in points:
        x = min(width - 1, max(0, int(point[0])))
        y = min(height - 1, max(0, int(point[1])))
        density[y, x] = 1

    density = gaussian_filter(density, sigma=4)

    h5_path = os.path.join(train_gt_dir, img_name.replace(".jpg", ".h5"))

    with h5py.File(h5_path, 'w') as hf:
        hf['density'] = density

print("Density map generation completed.")